<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Week2/Week_2_live_session.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤗 Hands-On Session: Tokenization with Hugging Face


---
## ⚙️ Setup — Install Libraries

In [ ]:
# Install all required libraries
!pip install transformers tokenizers datasets -q

---
## SEGMENT 1 — Why Tokenization? The Big Picture

A **tokenizer** splits input text into smaller units called **tokens**.
Tokens can be:
- **Words** → `["I", "love", "NLP"]`
- **Subwords** → `["token", "##ization"]`
- **Characters** → `["I", " ", "l", "o", "v", "e"]`

The Tokenizer Pipeline:
```
Raw Text
   ↓  Normalize (lowercase, strip accents)
   ↓  Pre-Tokenize (rough whitespace split)
   ↓  Tokenize (BPE / WordPiece / SentencePiece)
   ↓  Post-Process (add [CLS], [SEP] etc.)
   ↓
Token IDs  →  Embedding Matrix  →  Transformer Model
```

| Aspect                                        | Whitespace Level Tokenization       | Character Level Tokenization                    |
| --------------------------------------------- | ----------------------------------- | ----------------------------------------------- |
| Token Unit                                    | Words separated by spaces           | Individual characters                           |
| Example (`"I love AI"`)                       | `["I", "love", "AI"]`               | `["I", " ", "l", "o", "v", "e", " ", "A", "I"]` |
| Vocabulary Size                               | Large (all unique words)            | Very small (characters only)                    |
| Sequence Length                               | Short                               | Much longer                                     |
| Computational Cost                            | Lower                               | Higher                                          |
| Memory Usage                                  | Lower                               | Higher                                          |
| Out of Vocabulary (OOV) Problem               | Common for unseen words             | No OOV issue                                    |
| Handles Typos Well                            | Poorly                              | Better                                          |
| Semantic Information                          | Directly captures words             | Must learn words from characters                |
| Training Speed                                | Faster                              | Slower                                          |
| Attention Cost in Transformers                | Lower                               | Much higher due to longer sequences             |
| Multilingual Support                          | Difficult, vocabulary grows quickly | Easy, same character based approach             |
| Example with Unknown Word (`"ChatGPTxyz123"`) | Treated as a new unknown word       | Naturally tokenized into characters             |
| Used in Modern LLMs                           | Rarely                              | Rarely                                          |
| Main Limitation                               | Huge vocabulary, OOV words          | Long sequences, expensive computation           |


### Quick Comparison

```text id="br8u4m"
Sentence: "I love cryptocurrency"

Whitespace:
  ['I', 'love', 'cryptocurrency']

Character:
  ['I',' ','l','o','v','e',' ','c','r','y','p','t','o',...]

Subword:
  ['I', 'love', 'crypto', '##currency']
```

**Why modern LLMs prefer subword tokenization:**

```text id="ocm72z"
Whitespace:
  Short sequences ✓
  Huge vocabulary ✗
  OOV words ✗

Character:
  No OOV ✓
  Tiny vocabulary ✓
  Very long sequences ✗

Subword:
  Small vocabulary ✓
  Handles unseen words ✓
  Reasonable sequence lengths ✓

  → Best tradeoff, which is why BERT, GPT, Llama, Gemma,
    Mistral and most modern LLMs use subword tokenization.
```


```text id="j9wd1n"
BPE (Byte Pair Encoding) vs WordPiece
```

| Aspect                       | BPE                                               | WordPiece                                                      |
| ---------------------------- | ------------------------------------------------- | -------------------------------------------------------------- |
| Used By                      | GPT family, RoBERTa, Llama, Mistral, Gemma        | BERT, DistilBERT, ALBERT                                       |
| Starting Vocabulary          | Characters                                        | Characters                                                     |
| Main Idea                    | Merge the most frequent pair of tokens repeatedly | Add the token that maximizes likelihood of the training corpus |
| Merge Criterion              | Frequency based                                   | Probability based                                              |
| Training Objective           | Compress text by frequent merges                  | Build vocabulary that best explains the data                   |
| Complexity                   | Simpler                                           | More computationally expensive                                 |
| Result                       | Frequency driven vocabulary                       | Linguistically better vocabulary                               |
| Uses `##` Prefix             | No (typically)                                    | Yes                                                            |
| Speed of Vocabulary Creation | Faster                                            | Slower                                                         |
| Token Quality                | Good                                              | Often slightly better                                          |

---


```
BPE merges the most frequent adjacent token pairs repeatedly until the desired vocabulary size is reached. WordPiece also starts from characters but selects merges based on which merge maximizes the likelihood of the training corpus. Therefore, BPE is frequency driven, while WordPiece is probability driven. GPT style models commonly use BPE variants, whereas BERT uses WordPiece.
```


---
## SEGMENT 2 — Using Pre-Trained Tokenizers

### 2A — BERT Tokenizer (WordPiece)

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
tokenizer.vocab_size

30522

In [ ]:
tokenizer.model_max_length

512

In [ ]:
tokenizer.special_tokens_map

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

### 2B — GPT-2 Tokenizer (Byte-level BPE)

# **Comparison**

In [ ]:
# Side-by-side comparison
bert_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
print(f"{'BERT tokens':<40} {'GPT-2 tokens'}")
print("-" * 70)
for b, g in zip(bert_tokens, gpt2_tokens + [''] * max(0, len(bert_tokens)-len(gpt2_tokens))):
    print(f"  {b:<38} {g}")

print(f"\nBERT token count : {len(bert_tokens)}")
print(f"GPT-2 token count: {len(gpt2_tokens)}")

BERT tokens                              GPT-2 tokens
----------------------------------------------------------------------
  [CLS]                                  We
  we                                     Ġare
  are                                    Ġenjoying
  enjoying                               Ġthe
  the                                    Ġsession
  session                                
  [SEP]                                  

BERT token count : 7
GPT-2 token count: 5


In [ ]:
# ── Checking vocabulary size ───────────────────────────────────────────────
print(f"BERT  vocab_size : {tokenizer.vocab_size:,}")
print(f"GPT-2 vocab_size : {gpt_tokenizer.vocab_size:,}\n")

# Access first 10 vocab entries
vocab_items = list(tokenizer.get_vocab().items())
vocab_items_sorted = sorted(vocab_items, key=lambda x: x[1])  # sort by ID
print("First 10 entries in BERT vocab (sorted by ID):")
for tok, id_ in vocab_items_sorted[:10]:
    print(f"  ID {id_:>5}  →  '{tok}'")

BERT  vocab_size : 30,522
GPT-2 vocab_size : 50,257

First 10 entries in BERT vocab (sorted by ID):
  ID     0  →  '[PAD]'
  ID     1  →  '[unused0]'
  ID     2  →  '[unused1]'
  ID     3  →  '[unused2]'
  ID     4  →  '[unused3]'
  ID     5  →  '[unused4]'
  ID     6  →  '[unused5]'
  ID     7  →  '[unused6]'
  ID     8  →  '[unused7]'
  ID     9  →  '[unused8]'


---
## SEGMENT 3 — Building a Custom BPE Tokenizer from Scratch

When should you build a custom tokenizer?
- Domain-specific data (medical, legal, code)
- Training your own model from scratch
- Non-standard language / script
- Existing tokenizers don't perform well on your task

> **Rule:** Tokenizer must always match the model it's used with.

### 3A — Understanding BPE Algorithm

In [ ]:
# BPE Algorithm — Manual Illustration
from collections import Counter

# BPE starts with a character-level corpus
# </w> marks end of word
corpus = {
    "l o w </w>": 5,
    "l o w e r </w>": 2,
    "n e w e s t </w>": 6,
    "w i d e s t </w>": 3
}

def get_pair_counts(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

print("Initial corpus (characters + </w>):")
for w, f in corpus.items():
    print(f"  freq={f}  {w}")

pairs = get_pair_counts(corpus)
print("\nMost frequent adjacent pairs:")
for pair, freq in pairs.most_common(5):
    print(f"  {pair[0]} + {pair[1]}  →  freq={freq}")

best = pairs.most_common(1)[0]
print(f"\n Best merge: {best[0][0]} + {best[0][1]} = '{''.join(best[0])}'  (freq={best[1]})")
print("BPE repeats this thousands of times until target vocabulary size is reached.")

Initial corpus (characters + </w>):
  freq=5  l o w </w>
  freq=2  l o w e r </w>
  freq=6  n e w e s t </w>
  freq=3  w i d e s t </w>

Most frequent adjacent pairs:
  e + s  →  freq=9
  s + t  →  freq=9
  t + </w>  →  freq=9
  w + e  →  freq=8
  l + o  →  freq=7

 Best merge: e + s = 'es'  (freq=9)
BPE repeats this thousands of times until target vocabulary size is reached.


### 3B — Building the Pipeline (4 Steps)

In [ ]:
# All imports for custom tokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase, NFD, StripAccents, Sequence
from tokenizers.processors import TemplateProcessing
from tokenizers.decoders import BPEDecoder


In [ ]:
# ── Training corpus ────────────────────────────────────────────────────────
training_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "tokenization is the first step in natural language processing",
    "byte pair encoding merges the most frequent character pairs iteratively",
    "subword tokenization handles out of vocabulary words gracefully",
    "the transformer model relies on tokenized numerical inputs to the embedding matrix",
    "padding ensures all sequences in a batch have the same length for matrix operations",
    "truncation cuts sequences that exceed the maximum context length of the model",
    "the attention mask tells the model which tokens to attend and which to ignore",
    "special tokens like cls sep and mask are reserved in the vocabulary of bert models",
    "the vocabulary size is typically between thirty thousand and fifty thousand tokens",
    "encoding converts raw text to integer ids and decoding converts integer ids back to text",
    "running runner runs run all share the same root word in english morphology",
    "enjoyed enjoying enjoyable enjoy enjoyment all share the same stem enjoy",
    "natural language processing includes tokenization parsing tagging and text generation",
    "the embedding matrix maps each token id to a dense vector representation of fixed size",
    "softmax over the vocabulary gives the probability distribution for the next token prediction",
    "batch processing allows the model to learn from multiple training examples simultaneously",
    "the unknown token handles characters and words not seen during the training process",
    "normalization steps include lowercasing and removing diacritic marks from text",
    "pre tokenization performs a rough initial split before the subword algorithm is applied",
    "the hugging face tokenizers library makes it easy to build and train custom tokenizers",
    "word piece tokenization is used by bert while byte pair encoding is used by gpt models",
    "sentence piece tokenization works on raw text without requiring whitespace pre tokenization",
    "a smaller vocabulary size results in more tokens per sentence during tokenization",
    "character level tokenization produces very short vocabularies but very long sequences",
]

print(f" Corpus ready: {len(training_corpus)} sentences")
print(f" Total words : {sum(len(s.split()) for s in training_corpus)}")

 Corpus ready: 25 sentences
 Total words : 305


---
## Session Summary

```
╔══════════════════════════════════════════════════════════════════╗
║         TOKENIZATION — KEY TAKEAWAYS                            ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. Whitespace split  → vocab too large, OOV, fails on Japanese  ║
║  2. Character split   → tiny vocab, meaning lost, slow training  ║
║  3. Subword (BPE)     → sweet spot, 30k-50k vocab               ║
║                                                                  ║
║  4. ## prefix         → token is a CONTINUATION of previous     ║
║     ('HF' not in vocab → H + ##F  |  'let' in vocab → kept)     ║
║                                                                  ║
║  5. HF Pipeline: Normalize → Pre-tokenize → BPE → Post-process  ║
║                                                                  ║
║  6. Encoding returns 3 keys:                                     ║
║     input_ids | token_type_ids | attention_mask                  ║
║                                                                  ║
║  7. Padding  → uniform batch length (attention_mask=0 for pads)  ║
║  8. Truncation → enforce max_length context window limit         ║
║                                                                  ║
║  9. Special tokens: [UNK] [PAD] [CLS] [SEP] [MASK]              ║
║     → forced into vocab, not learned from data                   ║
║                                                                  ║
║ 10. BERT ↔ BERT tokenizer  |  GPT-2 ↔ GPT-2 tokenizer           ║
║     Always use the tokenizer that MATCHES the model!             ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
```